## Problem Setting
We have an asset price $(S_t)_{t \ge 0}$ following the Black-Scholes SDE, $\frac{dS_t}{S_t} = \mu dt + \sigma dW_t$ with $(W_t)_t$ the standard brownian motion.

We can expressed the price $V = (F(t,S_t))_{t \in [0,T], S_t \in \mathbb{R}}$ of an option with payoff $h(S_T)$ as following:

\begin{equation} 
    {\partial_t F} + r S {\partial_x F} + \frac{\sigma^{2}}{2}{S}^2 {\partial_xx F} - rF = 0 
\end{equation}
With $r$ the risk free rate of growth.

In the case of the general form of a semilinear parabolic PDE
\begin{equation}
    \partial_t u + \mu(t,x) \nabla_x u + \frac{1}{2}Tr(\sigma(t,x)[\sigma(t,x)]^{*}Hess_x(u)) + f(t,x,u,[\sigma(t,x)]^{*}\nabla_x u) = 0
\end{equation}

Here we're working on the particular case where:
\begin{equation}
    \begin{cases}
        u = V \\
        \mu(t,x) = rx \\
        \sigma(t,x) = \sigma x \\
        f(t,x,u, [\sigma(t,x)]^{*}\nabla_x u) = -ru \\ 
    \end{cases} \\
    \text{with the terminal condition } u(T,x) = g(x) 
\end{equation}

*In the case of a call option we would have $g(x) = (x - K)_{+}$ with $K$ the strike price.*


...


In [2]:
#imports
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sampling import brownian_increment, call_payoff, eu_call_price, basket_call_payoff, y0_init

#constants
mu = 0.05
r = 0.02
sigma = 0.2
T = 1

In [3]:
class SubNet(tf.keras.layers.Layer):
    def __init__(self, dim, num_hidden_layers=3, num_units=10):
        super().__init__()
        self.num_hidden_layers = num_hidden_layers
        self.dense_layers = [
            tf.keras.layers.Dense(num_units + dim, use_bias=False)
            for _ in range(num_hidden_layers - 1)
        ] + [tf.keras.layers.Dense(dim, use_bias=False)]
        self.bn_layers = [
            tf.keras.layers.BatchNormalization() for _ in range(num_hidden_layers + 1)
        ]

    def call(self, x, training=False):
        x = self.bn_layers[0](x, training=training)
        for i in range(self.num_hidden_layers - 1):
            x = self.dense_layers[i](x)
            x = self.bn_layers[i + 1](x, training=training)
            x = tf.nn.relu(x)
        x = self.dense_layers[-1](x)
        x = self.bn_layers[-1](x, training=training)
        return x


class DeepBSDE(tf.keras.Model):
    def __init__(self, d, N, S0, y0, z0, payoff_fn, r=0.02, sigma=0.2, rho=None, T=1.0):
        super().__init__()
        self.subnets = [SubNet(dim=d) for _ in range(N - 1)]

        self.y0 = self.add_weight(
            shape=(), initializer=tf.keras.initializers.Constant(y0),
            trainable=True, name="y0", dtype="float32"
        )
        self.z0 = self.add_weight(
            shape=(d,), initializer=tf.keras.initializers.Constant(z0),
            trainable=True, name="z0", dtype="float32"
        )

        self.S0 = tf.constant(S0, dtype=tf.float32)  

        sigma = tf.convert_to_tensor(sigma, dtype=tf.float32)
        if sigma.shape.rank == 0:
            sigma = tf.fill([d], sigma)
        self.sigma = sigma  

        self.rho = None if rho is None else tf.cast(rho, tf.float32)
        self.payoff_fn = payoff_fn  

        self.d, self.N, self.r, self.dt, self.T = d, N, r, T / N, T

    def call(self, dW, training=False):
        batch_size = tf.shape(dW)[0]
        X = tf.ones((batch_size, self.d)) * self.S0
        Y = tf.ones((batch_size, 1)) * self.y0
        Z = tf.ones((batch_size, self.d)) * self.z0

        for n in range(self.N):
            Y = Y + self.r * Y * self.dt \
                  + tf.reduce_sum(Z * dW[:, n, :], axis=1, keepdims=True)
            X = X + self.r * X * self.dt + self.sigma * X * dW[:, n, :]
            if n < self.N - 1:
                Z = self.subnets[n](X, training=training)

        return X, Y

    @tf.function
    def train_step(self, dW, optimizer):
        with tf.GradientTape() as tape:
            X, Y = self(dW, training=True)
            payoff = self.payoff_fn(X)
            loss = tf.reduce_mean((Y - payoff) ** 2)

        grads = tape.gradient(loss, self.trainable_variables)
        optimizer.apply_gradients(zip(grads, self.trainable_variables))
        return loss

    def train(self, optimizer=None, num_epochs=1000, batch_size=64, learning_rate=1e-3):
        lr_schedule = tf.keras.optimizers.schedules.PiecewiseConstantDecay(
            boundaries=[1000, 2000], values=[1e-2, 2e-3, 4e-4]
        )
        if optimizer is None:
            optimizer = tf.keras.optimizers.Adam(lr_schedule)
        else:
            optimizer = optimizer(lr_schedule)

        for epoch in range(num_epochs):
            dW = brownian_increment(batch_size, self.N, self.d, self.T, corr=self.rho)
            loss = self.train_step(dW, optimizer)
            if epoch % 100 == 0:
                tf.print("Epoch", epoch, "Loss:", loss, "y0: ", self.y0)

In [ ]:
S0_grid = [80, 90, 100, 110, 120]
K_grid  = [80, 100, 120, 140]

results = []
for S0 in S0_grid:
    for K in K_grid:
        m = DeepBSDE(d=1, N=50, X0=[float(S0)], y0=max(S0-K,0)*np.exp(-r*T),
                     z0=[20.0], K=float(K), r=0.05, sigma=0.2, T=1.0)
        m.train(num_epochs=2000, batch_size=1024)

        deep_price = m.y0.numpy()
        bs_price = eu_call_price(S0, K, T, r, sigma)
        results.append((S0, K, deep_price, bs_price, deep_price - bs_price))

for S0, K, dp, bp, err in results:
    print(f"S0={S0:>3} K={K:>3}  deep={dp:7.3f}  bs={bp:7.3f}  err={err:+.4f}")

# Results after 36m runtime

### S0 = 80

| K | deep | bs | err |
|---:|---:|---:|---:|
| 80 | 7.5800 | 7.1330 | +0.4470 |
| 100 | 1.8510 | 1.4270 | +0.4234 |
| 120 | 0.2820 | 0.1990 | +0.0833 |
| **140** | **0.0220** | **0.0220** | **-0.0001** |

### S0 = 90

| K | deep | bs | err |
|---:|---:|---:|---:|
| 80 | 15.6420 | 14.0230 | +1.6183 |
| 100 | 5.0500 | 4.1480 | +0.9021 |
| 120 | 1.1470 | 0.8600 | +0.2873 |
| 140 | 0.2070 | 0.1400 | +0.0673 |

### S0 = 100

| K | deep | bs | err |
|---:|---:|---:|---:|
| 80 | 24.5520 | 22.5430 | +2.0090 |
| 100 | 8.5670 | 8.9160 | -0.3486 |
| 120 | 3.2300 | 2.5470 | +0.6834 |
| 140 | 0.7670 | 0.5660 | +0.2011 |

### S0 = 110

| K | deep | bs | err |
|---:|---:|---:|---:|
| 80 | 34.1150 | 31.9290 | +2.1865 |
| 100 | 17.0560 | 15.6090 | +1.4470 |
| 120 | 6.6710 | 5.7560 | +0.9145 |
| 140 | 2.1670 | 1.6610 | +0.5056 |

### S0 = 120

| K | deep | bs | err |
|---:|---:|---:|---:|
| 80 | 43.9430 | 41.7000 | +2.2434 |
| 100 | 25.9460 | 23.7420 | +2.2041 |
| 120 | 9.2190 | 10.6990 | -1.4801 |
| 140 | 4.7970 | 3.8410 | +0.9561 |

**Best overall match:** S0 = 80, K = 140 (err = –0.0001)